# Sub-Area OD Matrices — 119-TAZ Study List

Extracts sub-matrices restricted to a given list of 119 TAZs (both origin **and** destination in the list) from the weighted Day 10 / Day 20 matrices — the all-mode versions and the four mode groups (CAR / TRANSIT / RAIL / OTHER).

Every sub-matrix is reindexed to the full 119-zone list **in the order given** (zones without observed trips become zero rows/columns), so all files share the same 119×119 shape and align cell-by-cell.

In [1]:
import numpy as np
import pandas as pd
import os

SUB_TAZ = [1520,2402,2412,2420,2405,2421,2404,2426,2413,2407,2406,2403,2415,1521,1524,2401,2408,
           1518,1519,1517,1516,1515,1514,1512,1510,1511,1501,1502,1215,1522,1503,1509,1508,1505,
           1506,1504,1302,1304,1507,1306,1225,1307,1312,1303,1308,1313,1309,1216,1217,1311,1310,
           1305,1401,1417,1408,1409,1726,1727,1218,1719,1212,1224,1214,1205,1213,1221,1220,1219,
           1211,1513,1523,1210,1209,1203,715,722,918,917,919,920,922,912,923,913,911,914,902,903,
           904,714,709,905,1222,408,422,423,409,1002,1005,1102,420,2101,1921,1906,2005,2001,2012,
           1904,2002,2010,2009,2007,2004,2008,2003,1924,1920,1801,1208]
assert len(SUB_TAZ) == len(set(SUB_TAZ)) == 119
SUB_TAZ_F = [float(t) for t in SUB_TAZ]

In [2]:
SOURCES = {
    'matrix_10_weighted.csv': 'day 10, all modes',
    'matrix_20_weighted.csv': 'day 20, all modes',
    'matrix_10_weighted_CAR.csv': 'day 10, CAR',
    'matrix_10_weighted_TRANSIT.csv': 'day 10, TRANSIT',
    'matrix_10_weighted_RAIL.csv': 'day 10, RAIL',
    'matrix_10_weighted_OTHER.csv': 'day 10, OTHER',
    'matrix_20_weighted_CAR.csv': 'day 20, CAR',
    'matrix_20_weighted_TRANSIT.csv': 'day 20, TRANSIT',
    'matrix_20_weighted_RAIL.csv': 'day 20, RAIL',
    'matrix_20_weighted_OTHER.csv': 'day 20, OTHER',
}

os.makedirs('Output/submatrices', exist_ok=True)
rows = []
for fname, label in SOURCES.items():
    m = pd.read_csv(f'Output/{fname}', index_col=0)
    m.index = m.index.astype(float)
    m.columns = m.columns.astype(float)
    sub = m.reindex(index=SUB_TAZ_F, columns=SUB_TAZ_F, fill_value=0.0)
    sub.index = SUB_TAZ
    sub.columns = SUB_TAZ
    sub.index.name = 'origin'
    sub.to_csv(f'Output/submatrices/{fname}')
    rows.append({'matrix': label,
                 'total expanded trips': m.sum().sum(),
                 'within sub-area': sub.sum().sum(),
                 'coverage': sub.sum().sum() / m.sum().sum()})

summary = pd.DataFrame(rows).set_index('matrix')
summary.round({'total expanded trips': 0, 'within sub-area': 0, 'coverage': 3})

,total expanded trips,within sub-area,coverage
matrix,,,
"day 10, all modes",2193422.0,142532.0,0.065
"day 20, all modes",2163320.0,139007.0,0.064
"day 10, CAR",1234371.0,64146.0,0.052
"day 10, TRANSIT",143215.0,13734.0,0.096
"day 10, RAIL",3990.0,0.0,0.000
"day 10, OTHER",811847.0,64653.0,0.080
"day 20, CAR",1202092.0,62758.0,0.052
"day 20, TRANSIT",143776.0,14291.0,0.099
"day 20, RAIL",4669.0,169.0,0.036


## Notes

- Cell (i, j) holds the expanded (`wf_new`-weighted) AM-peak trips from TAZ i to TAZ j, both inside the 119-zone list; trips with only one end inside the sub-area are excluded by construction.
- All ten files share the identical 119×119 layout in the given zone order — they can be added or compared cell-by-cell (the four mode files per day sum to that day's all-mode file).
- 16 of the 119 zones never appear as an AM-peak survey origin across the two days pooled (15 never as a destination); their rows/columns are structural zeros, not missing data.